In [1]:
##See README_WGS_analysis.txt for detailed info
##Input files:
##WGS BAM files (output from Bowtie2) aligned to hg19/GRCh37 filtered for reads from IG/TCR loci (WGS_align_to_hg19.ipynb)

import os
import re
import subprocess
import time
import pysam
import shlex
import io

## Full path to blast database, default is 'script_directory/blast_db/VDJ_RSS_extended.fasta' in the same folder as this script
blast_database = '/Volumes/4TBEncrypted/blast_db/VDJ_RSS_extended.fasta'

In [20]:
## Provide folder path of analysed patient BAM files:
folder_path = '/Volumes/4TBEncrypted/Downloads/Patients/PD3978a/'
os.chdir(folder_path)

patient_identifier = os.getcwd().split('/')[-1]
# dictionary is ORDERED: make sure order is V,J or V,D,J
tra = {}
tra['TRA_V_region'] = 'chr14:22,073,462-22,810,475'
tra['TRA_J_region'] = 'chr14:22,941,599-23,021,709'

trb = {}
trb['TRB_V_region'] = 'chr7:141,996,450-142,456,015'
# only 1 TRB D gene segment; very close to J region
trb['TRB_D_region'] = 'chr7:142,493,974-142,494,080'
trb['TRB_J_region'] = 'chr7:142,493,938-142,495,205'

trg = {}
trg['TRG_V_region'] = 'chr7:38,328,541-38,409,352'
trg['TRG_J_region'] = 'chr7:38,290,780-38,318,103'

trd = {}
trd['TRD_V_region'] = 'chr14:22,891,088-22,892,552'
trd['TRD_D_region'] = 'chr14:22,906,997-22,918,659'
trd['TRD_J_region'] = 'chr14:22,918,659-22,928,637'

igh = {}
igh['IGH_V_region'] = 'chr14:106,394,586-107,305,139'
igh['IGH_D_region'] = 'chr14:106,343,168-106,390,040'
igh['IGH_J_region'] = 'chr14:106,327,095-106,334,338'

igk = {}
igk['IGK_V_region'] = 'chr2:89,175,118-90,290,685'
# include KDE:
igk['IGK_J_region'] = 'chr2:89,131,163-89,163,461'

igl = {}
igl['IGL_V_region'] = 'chr22:22,375,231-23,226,511'
igl['IGL_J_region'] = 'chr22:23,231,745-23,267,392'

locus_list = [tra,trb,trg,trd,igh,igk,igl]

V_J_loci = ['TRA','TRG','IGK','IGL']
V_D_J_loci = ['TRB','TRD','IGH']

positive_ori_locus = ['TRA','TRB','TRD','IGL']

filtered_read_ids = []

for locus_region in locus_list:
    locus_regions = list(locus_region.keys())
    locus_name = locus_regions[0].split('_')[0]
    locus_coord = list(locus_region.values())
    
    if locus_name in V_J_loci:

        pysam.index(f'{patient_identifier}.VDJ.bam',catch_stdout=False)
        pysam.view('-o','regionV_tmp.sam',f'{patient_identifier}.VDJ.bam',locus_coord[0],catch_stdout=False)
        pysam.view('-o','regionJ_tmp.sam',f'{patient_identifier}.VDJ.bam',locus_coord[1],catch_stdout=False)
        
        with open('regionV_tmp.sam','r') as rV:
            if locus_name in positive_ori_locus:
                reads_id_V = [line.split('\t')[0] for line in rV if line.split('\t')[1] in ('81','145')]
            else:
                reads_id_V = [line.split('\t')[0] for line in rV if line.split('\t')[1] in ('97','161')]
                
        with open('regionJ_tmp.sam','r') as rJ:
            if locus_name in positive_ori_locus:
                reads_id_J = [line.split('\t')[0] for line in rJ if line.split('\t')[1] in ('97','161')]
            else:
                reads_id_J = [line.split('\t')[0] for line in rJ if line.split('\t')[1] in ('81','145')]
                
        for id in reads_id_V:
            if id in reads_id_J:
                filtered_read_ids.append(id)
        
        # Delete temp files
        os.remove('regionV_tmp.sam')
        os.remove('regionJ_tmp.sam')
       
    if locus_name in V_D_J_loci:
        pysam.index(f'{patient_identifier}.VDJ.bam',catch_stdout=False)
        pysam.view('-o','regionV_tmp.sam',f'{patient_identifier}.VDJ.bam',locus_coord[0],catch_stdout=False)
        pysam.view('-o','regionD_tmp.sam',f'{patient_identifier}.VDJ.bam',locus_coord[1],catch_stdout=False)
        pysam.view('-o','regionJ_tmp.sam',f'{patient_identifier}.VDJ.bam',locus_coord[2],catch_stdout=False)
        
        with open('regionV_tmp.sam','r') as rV:
            if locus_name in positive_ori_locus:
                reads_id_V = [line.split('\t')[0] for line in rV if line.split('\t')[1] in ('81','145')]
            else:
                reads_id_V = [line.split('\t')[0] for line in rV if line.split('\t')[1] in ('97','161')]
            
        with open('regionD_tmp.sam','r') as rVD:
            if locus_name in positive_ori_locus:
                reads_id_VD = [line.split('\t')[0] for line in rVD if line.split('\t')[1] in ('97','161')]
            else:
                reads_id_VD = [line.split('\t')[0] for line in rVD if line.split('\t')[1] in ('81','145')]
                
        with open('regionD_tmp.sam','r') as rDJ:
            if locus_name in positive_ori_locus:
                reads_id_DJ = [line.split('\t')[0] for line in rDJ if line.split('\t')[1] in ('81','145')]
            else:
                reads_id_DJ = [line.split('\t')[0] for line in rDJ if line.split('\t')[1] in ('97','161')]
        
        with open('regionJ_tmp.sam','r') as rJ:
            if locus_name in positive_ori_locus:
                reads_id_J = [line.split('\t')[0] for line in rJ if line.split('\t')[1] in ('97','161')]
            else:
                reads_id_J = [line.split('\t')[0] for line in rJ if line.split('\t')[1] in ('81','145')]
            
        for id in reads_id_V:
            if id in reads_id_J:
                filtered_read_ids.append(id)
                
        for id in reads_id_V:
            if id in reads_id_VD:
                filtered_read_ids.append(id)
                
        for id in reads_id_DJ:
            if id in reads_id_J:
                filtered_read_ids.append(id)
        
        # Delete temp files
        os.remove('regionV_tmp.sam')
        os.remove('regionD_tmp.sam')
        os.remove('regionJ_tmp.sam')

        
# # Extract header from bam file
bam_header = pysam.view('-H',f'{patient_identifier}.VDJ.bam')

pysam.view('-o','tmp.sam',f'{patient_identifier}.VDJ.bam',catch_stdout=False)

with open('tmp.sam','r') as sam_file:
        extracted_reads = [line for line in sam_file if line.split('\t')[0] in set(filtered_read_ids)]

samfile_output = bam_header + ''.join(extracted_reads)
    
with open('tmp2.sam','w') as output_file:
     output_file.write(samfile_output)
    
output_bam = f'{patient_identifier}.VDJ.ESC_filtered4.bam'
    
pysam.sort('-o',output_bam,'tmp2.sam',catch_stdout=False)
pysam.index(output_bam,catch_stdout=False)

# Delete temp files
os.remove('tmp.sam')
os.remove('tmp2.sam')

In [21]:
# Blast


BLAST_OUTFMT6 = """\
    '6 qacc sacc pident length mismatch gapopen qstart qend sstart send evalue bitscore qseq sseq'\
"""

BLAST_OUTFMT6_COLUMN_NAMES = [
    'query_id', 'subject_id', 'pc_identity', 'alignment_length', 'mismatches', 'gap_opens',
    'q_start', 'q_end', 's_start', 's_end', 'evalue', 'bitscore', 'qseq', 'sseq',
]

def blastn(sequence, db=blast_database, evalue=0.001, max_target_seqs=100000):
    system_command = (
        'blastn -db {db} -outfmt {outfmt} -evalue {evalue} -max_target_seqs {max_target_seqs} -task blastn-short'
        .format(db=db, outfmt=BLAST_OUTFMT6, evalue=evalue, max_target_seqs=max_target_seqs)
    )
    cp = subprocess.Popen(
        shlex.split(system_command),
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        universal_newlines=True)
    result, error_message = cp.communicate(sequence)
    if error_message.strip():
        pass
        #print("Error: {}".format(error_message))
    return result


pysam.view('-o','tmp.sam',f'{patient_identifier}.VDJ.ESC_filtered4.bam',catch_stdout=False)
        
with open('tmp.sam','r') as sam_file:
    reads = [line for line in sam_file]
            
os.remove('tmp.sam')

#with header:
output = ['Patient_ID\tRead_ID\tSequence\tSequence_length\tRSS\tRSS_region_coordinate\tBLAST_score\tBLAST_match']
for read in reads:
    read_id = read.split('\t')[0]
    fasta_seq = read.split('\t')[9]
    result = blastn(fasta_seq,evalue=0.001,max_target_seqs=3)
    for x in range(1,len(result.split('Query_1'))):
        result_RSS = result.split('Query_1')[x].split('\t')[1]
        result_score = result.split('Query_1')[x].split('\t')[2]
        result_match = result.split('Query_1')[x].split('\t')[3]
        #if int(result_match) >= len(fasta_seq)*0.7:
        #output.append(f'{patient_identifier}\t{read_id}\t{fasta_seq}\t{len(fasta_seq)}\t{(result_RSS.split("::")[0]).split("RSS")[1]}\t{result_RSS.split("::")[1]}\t{result_score}\t{result_match}')
        output.append(f'{patient_identifier}\t{read_id}\t{fasta_seq}\t{len(fasta_seq)}\t{(result_RSS.split("::")[0])}\t{result_RSS.split("::")[1]}\t{result_score}\t{result_match}')
tsv_output = ('\n'.join(output))
with open(f'{patient_identifier}_ESC_analysis.tsv','w') as f:
    f.write(tsv_output)